# MetaJudge: Evidence-Assisted Self-Correction (C2)

**Task:** `metajudge_sc_c2` | **Version:** v4.1
**Axis:** Control (Nelson & Narens 1990)
**Items:** 23 clean C2 items
**Score:** Anchor-normalized T2-T1 accuracy delta → float in [0, 1]

In [ ]:
import sys, os, json
sys.path.insert(0, "/kaggle/input/metajudge-package-v3")
DATA_ROOT = "/kaggle/input/metajudge-data-v3"

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C2_T2_TEMPLATE, parse_answer_confidence,
    compute_edit_similarity, resolve_t2_answer,
)
from metajudge.scoring.self_correction_v2 import classify_transition

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
import numpy as np

ANCHOR_C2_FLOOR = -0.2   # Empirical: min - 2*SD across all model-runs
ANCHOR_C2_CEIL  = 0.2   # Empirical: max + 2*SD across all model-runs

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

print(f"Scoring: transition-based (package), anchor C2=[{ANCHOR_C2_FLOOR}, {ANCHOR_C2_CEIL}]")


In [ ]:
# (no schema needed — uses text parsing)


In [ ]:
import pandas as pd

with open(os.path.join(DATA_ROOT, "family_c_items.json")) as f:
    all_fc_items = json.load(f)

with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
fc_items = [it for it in all_fc_items if it["item_id"] not in fc_excluded]
c2_items = [it for it in fc_items if it.get("subfamily") == "C2"]
c2_df = pd.DataFrame(c2_items)

REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))


In [ ]:
@kbench.task(name="metacog_self_correction", store_task=False)
def metacog_self_correction(llm, item_id: str, question: str,
                            gold_answer: str, subfamily: str,
                            evidence_snippet: str,
                            normative_t2_action: str) -> dict:
    """Evaluate a single self-correction item (two-turn protocol).

    Turn 1: Model answers the question with ANSWER + CONFIDENCE.
    Turn 2: Model receives review prompt and may revise.
      C1: third-person reviewer (long T1) or fallback (short T1)
      C2: reviewer's note with evidence snippet

    Returns dict with item-level audit data.
    """
    with chats.new():
        # Turn 1
        t1_resp = llm.prompt(question + T1_SUFFIX)
        t1_text = str(t1_resp)
        t1_answer, t1_conf = parse_answer_confidence(t1_text)

        # Turn 2
        if subfamily == "C1":
            t2_prompt = (C1_T2_PRIMARY if len(t1_text) > C1_PRIMARY_MIN_LENGTH
                         else C1_T2_FALLBACK)
        else:  # C2
            t2_prompt = C2_T2_TEMPLATE.format(
                evidence=evidence_snippet or "")
        t2_resp = llm.prompt(t2_prompt)
        t2_text = str(t2_resp)
        t2_answer, t2_conf = parse_answer_confidence(t2_text)

    # Resolve T2 answer (handle confirmation-without-restatement)
    t2_answer_resolved = resolve_t2_answer(t2_answer, t1_answer, gold_answer)

    # Grade both turns
    t1_correct = grade_item(item_id, t1_answer, REGISTRY).get("correct", False)
    t2_correct = grade_item(item_id, t2_answer_resolved, REGISTRY).get("correct", False)

    # Classify transition
    revised = t1_answer.strip().lower() != t2_answer.strip().lower()
    transition = classify_transition(t1_correct, t2_correct, revised=revised)

    # Edit distance
    edit_sim = compute_edit_similarity(t1_text, t2_text)

    return {
        "item_id": item_id,
        "subfamily": subfamily,
        "gold_answer": gold_answer,
        "t1_answer": t1_answer[:200],
        "t2_answer": t2_answer[:200],
        "t1_confidence": round(t1_conf, 4) if t1_conf is not None else None,
        "t2_confidence": round(t2_conf, 4) if t2_conf is not None else None,
        "t1_correct": t1_correct,
        "t2_correct": t2_correct,
        "transition": transition,
        "t1_t2_similarity": round(edit_sim, 4),
        "normative_t2_action": normative_t2_action,
    }


In [ ]:
@kbench.task(name="metajudge_sc_c2")
def metajudge_sc_c2(llm) -> float:
    """Family C2 — Evidence-Assisted Self-Correction (C2) (Control).
    
    Returns anchor-normalized T2-T1 accuracy delta.
    Includes dual-run stochasticity check (Run 2 is display only, not scored).
    """
    sub_df = fc_df[fc_df["subfamily"] == "C2"][
        ["item_id", "question", "gold_answer", "subfamily",
         "evidence_snippet", "normative_t2_action"]
    ].copy()
    sub_df["evidence_snippet"] = sub_df["evidence_snippet"].fillna("")

    # Run 1 (scored)
    with kbench.client.enable_cache():
        runs_1 = metacog_self_correction.evaluate(
            llm=[llm], evaluation_data=sub_df,
            n_jobs=4, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(sub_df),
            max_attempts=1,
        )
    records_1 = [r.result for r in runs_1 if r.result is not None]

    # Run 2 (stochasticity check — display only)
    runs_2 = metacog_self_correction.evaluate(
        llm=[llm], evaluation_data=sub_df,
        n_jobs=4, remove_run_files=True,
        stop_condition=lambda runs: len(runs) == len(sub_df),
        max_attempts=1,
    )
    records_2 = [r.result for r in runs_2 if r.result is not None]

    # Score from Run 1
    audit = pd.DataFrame(records_1)
    t1c = audit["t1_correct"].sum()
    t2c = audit["t2_correct"].sum()
    delta = float((t2c - t1c) / len(audit))
    normalized = normalize(delta, ANCHOR_C2_FLOOR, ANCHOR_C2_CEIL)

    # Stochasticity comparison
    audit_2 = pd.DataFrame(records_2)
    t1c_2 = audit_2["t1_correct"].sum()
    t2c_2 = audit_2["t2_correct"].sum()
    delta_2 = float((t2c_2 - t1c_2) / len(audit_2))
    norm_2 = normalize(delta_2, ANCHOR_C2_FLOOR, ANCHOR_C2_CEIL)

    # Transition stability
    r1_sorted = sorted(records_1, key=lambda r: r["item_id"])
    r2_sorted = sorted(records_2, key=lambda r: r["item_id"])
    trans_diffs = sum(1 for a, b in zip(r1_sorted, r2_sorted)
                      if a["transition"] != b["transition"])
    stable = len(audit) - trans_diffs

    # Damage stats
    t1_right = audit["t1_correct"].sum()
    rw = ((audit["t1_correct"]) & (~audit["t2_correct"])).sum()
    dmg_rate = rw / t1_right if t1_right > 0 else 0.0

    print(f"C2: Δ={delta:+.4f} Norm={normalized:.3f} Dmg={dmg_rate:.1%} [{len(audit)} items]")
    print(f"  Stochasticity: Run 1 Δ={delta:+.4f}, Run 2 Δ={delta_2:+.4f}")
    print(f"  Transition stability: {stable}/{len(audit)} items stable ({stable/len(audit):.0%})")
    print(f"  \u2192 Score ranges from {min(normalized, norm_2):.2f} to {max(normalized, norm_2):.2f} across independent runs")

    # Export
    audit.to_csv(os.path.join(OUTPUT_DIR, "family_c2_item_audit.csv"), index=False)
    import json as _json
    with open(os.path.join(OUTPUT_DIR, "family_c2_full_responses.json"), "w") as f:
        _json.dump({"run_1": records_1, "run_2": records_2}, f, indent=2, default=str)

    return normalized


In [ ]:
metajudge_sc_c2.run(kbench.llm)
%choose metajudge_sc_c2


## Methodology

**Axis IV — Evidence-Assisted Self-Correction (Control).** Two-turn protocol: T1 answer →
reviewer note with evidence snippet → T2 revision. 23 clean C2 items. Tests whether external
evidence improves or damages performance. Confirmation-without-restatement detection applied to
T2 answers. Dual-run stochasticity check displayed in output. Normalized using empirically
grounded anchors (floor=-0.20, ceil=+0.20) derived from 19 model-run observations.

Equal weighting across tasks is provably optimal at small n (Davis-Stober 2011; Dawes 1979).
Anchor normalization follows BIG-Bench (Srivastava et al. 2023).